# Checkpoint 1: Multi-Agent Reinforcement Learning Experiments

Independent Q-learning VS Cooperative Q-learning


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
from collections import deque
import time
import random
import math
from collections import defaultdict

# 设置随机种子以确保结果可复现
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

try:
    import gymnasium as gym
    from gymnasium import spaces
except ImportError:
    # Fallback to gym if gymnasium is not available
    import gym  # type: ignore
    from gym import spaces  # type: ignore


In [2]:
# ============================================
# 第一部分：基础环境配置和类
# ============================================

class WarehouseConfig:
    """仓库环境配置类"""
    def __init__(self):
        self.grid_size = (6, 6)
        self.start_pos = None
        self.pickup_pos = (0, 5)
        self.dropoff_pos = (5, 0)
        self.obstacles = set([
            (1, 1), (1, 2), (1, 3), 
            (3, 1), (3, 2), (3, 3),  
            (4, 4), (5, 4)           
        ])
        # 优化后的奖励设计（平衡效率和探索）
        self.step_penalty = -0.8  # 适度惩罚步数，鼓励效率
        self.obstacle_penalty = -15.0  # 保持障碍物惩罚
        self.first_pick_reward = 35.0  # 增加拾取奖励
        self.delivery_reward = 200.0  # 大幅增加交付奖励，强化任务完成
        self.render_mode = "matplotlib"


class WarehouseBase(gym.Env):
    """基础仓库环境类"""
    
    def __init__(self, config=None, seed=None):
        gym.Env.__init__(self)
        if config is None:
            config = WarehouseConfig()
        self.cfg = config
        self.rows, self.cols = self.cfg.grid_size
        self.obstacles = set(self.cfg.obstacles)
        self.pickup = self.cfg.pickup_pos
        self.dropoff = self.cfg.dropoff_pos
        self.observation_space = spaces.MultiDiscrete([self.rows, self.cols, 2])
        self.action_space = spaces.Discrete(6)
        self.rng = np.random.default_rng(seed)
        self.state = (0, 0)
        self.carrying = False
        self.picked_once = False
        self.delivered = False
        self.total_steps = 0
        self.total_reward = 0.0

    def get_valid_start_positions(self):
        valid_positions = []
        for r in range(self.rows):
            for c in range(self.cols):
                pos = (r, c)
                if (pos not in self.obstacles and 
                    pos != self.pickup and 
                    pos != self.dropoff):
                    valid_positions.append(pos)
        return valid_positions

    def move(self, action):
        UP, DOWN, LEFT, RIGHT, PICK, DROP = 0, 1, 2, 3, 4, 5
        r, c = self.state
        if action == UP:
            return (max(0, r - 1), c)
        elif action == DOWN:
            return (min(self.rows - 1, r + 1), c)
        elif action == LEFT:
            return (r, max(0, c - 1))
        elif action == RIGHT:
            return (r, min(self.cols - 1, c + 1))
        return self.state

    def apply_move_with_collisions(self, target_position):
        if target_position in self.obstacles:
            return self.state, self.cfg.obstacle_penalty
        return (target_position, 0.0)

    def handle_pickup(self):
        if isinstance(self.state, tuple):
            robot_position = self.state
        else:
            robot_position = tuple(self.state)
        if robot_position != self.pickup:
            return 0.0
        if self.carrying:
            return 0.0
        self.carrying = True
        if not self.picked_once:
            self.picked_once = True
            return self.cfg.first_pick_reward
        return 0.0

    def handle_dropoff(self):
        if isinstance(self.state, tuple):
            robot_position = self.state
        else:
            robot_position = tuple(self.state)
        if not self.carrying:
            return (0.0, False)
        if robot_position != self.dropoff:
            return (0.0, False)
        self.carrying = False
        self.delivered = True
        return (self.cfg.delivery_reward, True)

    def reset(self, seed=None, options=None):
        gym.Env.reset(self, seed=seed)
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        valid_positions = self.get_valid_start_positions()
        if valid_positions:
            self.state = tuple(self.rng.choice(valid_positions))
        else:
            self.state = (0, 0)
        self.carrying = False
        self.picked_once = False
        self.delivered = False
        self.total_steps = 0
        self.total_reward = 0.0
        obs = np.array([self.state[0], self.state[1], int(self.carrying)], dtype=np.int64)
        return obs, {"position": self.state, "carrying": self.carrying}

    def step(self, action):
        UP, DOWN, LEFT, RIGHT, PICK, DROP = 0, 1, 2, 3, 4, 5
        step_reward = self.cfg.step_penalty
        game_ended = False
        game_truncated = False
        if action in (UP, DOWN, LEFT, RIGHT):
            target_position = self.move(action)
            actual_position, collision_penalty = self.apply_move_with_collisions(target_position)
            self.state = actual_position
            step_reward += collision_penalty
        elif action == PICK:
            pickup_reward = self.handle_pickup()
            step_reward += pickup_reward
        elif action == DROP:
            dropoff_reward, completed = self.handle_dropoff()
            step_reward += dropoff_reward
            if completed:
                game_ended = True
        self.total_steps += 1
        self.total_reward += step_reward
        if self.total_steps >= 200:
            game_truncated = True
        obs = np.array([self.state[0], self.state[1], int(self.carrying)], dtype=np.int64)
        info = {
            "position": self.state,
            "carrying": self.carrying,
            "delivered": self.delivered,
            "total_steps": self.total_steps
        }
        return obs, step_reward, game_ended, game_truncated, info

    def render(self):
        pass

    def close(self):
        pass


class WarehouseRobotDetEnv(WarehouseBase):
    """确定性仓库机器人环境（用于多智能体环境的基础环境）"""
    pass


In [3]:
# ============================================
# 第二部分：多智能体环境
# ============================================

class MultiAgentWarehouseEnv(gym.Env):
    """多智能体仓库机器人环境，支持N个智能体同时执行任务"""
    
    def __init__(self, base_env_class, config=None, num_agents=3, seed=None):
        gym.Env.__init__(self)
        self.num_agents = num_agents
        self.base_env_class = base_env_class
        if config is None:
            config = WarehouseConfig()
        self.cfg = config
        self.rows, self.cols = self.cfg.grid_size
        self.obstacles = set(self.cfg.obstacles)
        self.pickup = self.cfg.pickup_pos
        self.dropoff = self.cfg.dropoff_pos
        self.rng = np.random.default_rng(seed)
        self.agent_envs = []
        for i in range(num_agents):
            env = base_env_class(config=config, seed=seed)
            self.agent_envs.append(env)
        self.observation_space = spaces.Tuple([
            spaces.MultiDiscrete([self.rows, self.cols, 2]) 
            for _ in range(num_agents)
        ])
        self.action_space = spaces.Tuple([
            spaces.Discrete(6) for _ in range(num_agents)
        ])
        self.agent_states = [(0, 0) for _ in range(num_agents)]
        self.agent_carrying = [False for _ in range(num_agents)]
        self.agent_picked_once = [False for _ in range(num_agents)]
        self.agent_delivered = [False for _ in range(num_agents)]
        self.total_steps = 0
        
    def get_valid_start_positions(self):
        valid_positions = []
        for r in range(self.rows):
            for c in range(self.cols):
                pos = (r, c)
                if (pos not in self.obstacles and 
                    pos != self.pickup and 
                    pos != self.dropoff):
                    valid_positions.append(pos)
        return valid_positions
    
    def reset(self, seed=None, options=None):
        gym.Env.reset(self, seed=seed)
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        observations = []
        for i, env in enumerate(self.agent_envs):
            obs, info = env.reset(seed=seed)
            # 确保位置是tuple类型
            pos_tuple = tuple(obs[:2])
            self.agent_states[i] = pos_tuple
            self.agent_carrying[i] = bool(obs[2])
            self.agent_picked_once[i] = False
            self.agent_delivered[i] = False
            observations.append(obs)
        self._resolve_start_position_conflicts()
        observations = []
        for i in range(self.num_agents):
            # 确保agent_states[i]是tuple
            pos = self.agent_states[i]
            if isinstance(pos, np.ndarray):
                pos = tuple(pos)
            elif not isinstance(pos, tuple):
                pos = tuple(pos)
            self.agent_states[i] = pos
            
            obs = np.array([
                pos[0],
                pos[1],
                int(self.agent_carrying[i])
            ], dtype=np.int64)
            observations.append(obs)
        self.total_steps = 0
        info = {
            "agent_positions": [self.agent_states[i] for i in range(self.num_agents)],
            "agent_carrying": [self.agent_carrying[i] for i in range(self.num_agents)],
            "agent_delivered": [self.agent_delivered[i] for i in range(self.num_agents)]
        }
        return tuple(observations), info
    
    def _resolve_start_position_conflicts(self):
        used_positions = set()
        valid_positions = self.get_valid_start_positions()
        for i in range(self.num_agents):
            # 确保位置是tuple类型（可哈希）
            current_pos = self.agent_states[i]
            if isinstance(current_pos, np.ndarray):
                current_pos = tuple(current_pos)
            elif not isinstance(current_pos, tuple):
                current_pos = tuple(current_pos)
            
            if current_pos in used_positions:
                available = [p for p in valid_positions if p not in used_positions]
                if available:
                    new_pos = self.rng.choice(available)
                    # 确保new_pos是tuple类型
                    if isinstance(new_pos, np.ndarray):
                        new_pos = tuple(new_pos)
                    elif not isinstance(new_pos, tuple):
                        new_pos = tuple(new_pos)
                    self.agent_states[i] = new_pos
                    self.agent_envs[i].state = new_pos
                    current_pos = new_pos  # 更新current_pos为新的位置
            
            # 再次确保current_pos是tuple（防止任何意外情况）
            if isinstance(current_pos, np.ndarray):
                current_pos = tuple(current_pos)
            elif not isinstance(current_pos, tuple):
                current_pos = tuple(current_pos)
            
            used_positions.add(current_pos)
    
    def _handle_agent_collisions(self, proposed_positions):
        final_positions = list(proposed_positions)
        position_counts = {}
        for i, pos in enumerate(final_positions):
            pos_tuple = tuple(pos)
            if pos_tuple not in position_counts:
                position_counts[pos_tuple] = []
            position_counts[pos_tuple].append(i)
        for pos_tuple, agent_indices in position_counts.items():
            if len(agent_indices) > 1:
                for agent_idx in agent_indices:
                    final_positions[agent_idx] = self.agent_states[agent_idx]
        return final_positions
    
    def step(self, actions):
        UP, DOWN, LEFT, RIGHT, PICK, DROP = 0, 1, 2, 3, 4, 5
        
        def clamp(value, min_val, max_val):
            return max(min_val, min(value, max_val))
        
        self.total_steps += 1
        individual_rewards = []
        proposed_positions = []
        for i, action in enumerate(actions):
            reward = self.cfg.step_penalty
            if action in (UP, DOWN, LEFT, RIGHT):
                r, c = self.agent_states[i]
                if action == UP:
                    r -= 1
                elif action == DOWN:
                    r += 1
                elif action == LEFT:
                    c -= 1
                elif action == RIGHT:
                    c += 1
                r = clamp(r, 0, self.rows - 1)
                c = clamp(c, 0, self.cols - 1)
                proposed_pos = (r, c)
                if proposed_pos in self.obstacles:
                    reward += self.cfg.obstacle_penalty
                    proposed_pos = self.agent_states[i]
                proposed_positions.append(proposed_pos)
                individual_rewards.append(reward)
            else:
                proposed_positions.append(self.agent_states[i])
                individual_rewards.append(self.cfg.step_penalty)
        final_positions = self._handle_agent_collisions(proposed_positions)
        for i, action in enumerate(actions):
            self.agent_states[i] = final_positions[i]
            self.agent_envs[i].state = final_positions[i]
            if action == PICK:
                reward = self._handle_pickup(i)
                individual_rewards[i] += reward
            elif action == DROP:
                reward, is_completed = self._handle_dropoff(i)
                individual_rewards[i] += reward
                if is_completed:
                    self.agent_delivered[i] = True
        all_delivered = all(self.agent_delivered)
        terminated = all_delivered
        observations = []
        for i in range(self.num_agents):
            obs = np.array([
                self.agent_states[i][0],
                self.agent_states[i][1],
                int(self.agent_carrying[i])
            ], dtype=np.int64)
            observations.append(obs)
        team_reward = sum(individual_rewards)
        info = {
            "individual_rewards": individual_rewards,
            "team_reward": team_reward,
            "agent_positions": [self.agent_states[i] for i in range(self.num_agents)],
            "agent_carrying": [self.agent_carrying[i] for i in range(self.num_agents)],
            "agent_delivered": [self.agent_delivered[i] for i in range(self.num_agents)],
            "total_steps": self.total_steps
        }
        return (
            tuple(observations),
            individual_rewards,
            terminated,
            False,
            info
        )
    
    def _handle_pickup(self, agent_idx):
        agent_pos = self.agent_states[agent_idx]
        if agent_pos != self.pickup:
            return 0.0
        if self.agent_carrying[agent_idx]:
            return 0.0
        self.agent_carrying[agent_idx] = True
        self.agent_envs[agent_idx].carrying = True
        if not self.agent_picked_once[agent_idx]:
            self.agent_picked_once[agent_idx] = True
            return self.cfg.first_pick_reward
        return 0.0
    
    def _handle_dropoff(self, agent_idx):
        agent_pos = self.agent_states[agent_idx]
        if not self.agent_carrying[agent_idx]:
            return (0.0, False)
        if agent_pos != self.dropoff:
            return (0.0, False)
        self.agent_carrying[agent_idx] = False
        self.agent_envs[agent_idx].carrying = False
        self.agent_delivered[agent_idx] = True
        return (self.cfg.delivery_reward, True)
    
    def close(self):
        for env in self.agent_envs:
            env.close()


In [4]:
# ============================================
# 第三部分：Independent Q-Learning算法
# ============================================

class IndependentQLearning:
    """独立Q学习算法：每个智能体独立学习，不共享Q表"""
    
    def __init__(self, num_agents, state_space_size, action_space_size, 
                 learning_rate=0.1, discount_factor=0.95, epsilon_start=1.0, 
                 epsilon_end=0.1, epsilon_decay=0.995, seed=42):
        self.num_agents = num_agents
        self.state_space_size = state_space_size
        self.action_space_size = action_space_size
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.epsilon = epsilon_start
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        # 使用固定的随机数生成器以确保可复现
        self.rng = np.random.default_rng(seed)
        self.random_state = random.getstate()
        random.seed(seed)
        self.q_tables = [defaultdict(lambda: np.zeros(action_space_size)) 
                         for _ in range(num_agents)]
        self.episode_rewards = [[] for _ in range(num_agents)]
        self.total_updates = [0 for _ in range(num_agents)]
    
    def state_to_key(self, state):
        return tuple(state.astype(int))
    
    def select_action(self, agent_idx, state, training=True):
        state_key = self.state_to_key(state)
        if training and self.rng.random() < self.epsilon:
            return self.rng.integers(0, self.action_space_size)
        else:
            q_values = self.q_tables[agent_idx][state_key]
            max_q = np.max(q_values)
            best_actions = np.where(q_values == max_q)[0]
            return int(self.rng.choice(best_actions))
    
    def update(self, agent_idx, state, action, reward, next_state, done):
        state_key = self.state_to_key(state)
        next_state_key = self.state_to_key(next_state)
        current_q = self.q_tables[agent_idx][state_key][action]
        if done:
            target_q = reward
        else:
            next_max_q = np.max(self.q_tables[agent_idx][next_state_key])
            target_q = reward + self.discount_factor * next_max_q
        new_q = current_q + self.learning_rate * (target_q - current_q)
        self.q_tables[agent_idx][state_key][action] = new_q
        self.total_updates[agent_idx] += 1
    
    def decay_epsilon(self, episode):
        """指数衰减epsilon"""
        self.epsilon = max(self.epsilon_end, 
                          self.epsilon_start * math.exp(-self.epsilon_decay * episode))
    
    def get_policy(self, agent_idx, state):
        state_key = self.state_to_key(state)
        q_values = self.q_tables[agent_idx][state_key]
        return int(np.argmax(q_values))


In [5]:
# ============================================
# 第四部分：Cooperative Q-Learning算法
# ============================================

class CooperativeQLearning:
    """合作Q学习算法：所有智能体共享一个Q表，使用联合状态和团队奖励"""
    
    def __init__(self, num_agents, state_space_size, action_space_size,
                 learning_rate=0.1, discount_factor=0.95, epsilon_start=1.0,
                 epsilon_end=0.1, epsilon_decay=0.995, seed=42):
        self.num_agents = num_agents
        self.state_space_size = state_space_size
        self.action_space_size = action_space_size
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.epsilon = epsilon_start
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        # 使用固定的随机数生成器以确保可复现
        self.rng = np.random.default_rng(seed)
        self.joint_action_space_size = action_space_size ** num_agents
        self.q_table = defaultdict(lambda: np.zeros(self.joint_action_space_size))
        self.episode_rewards = []
        self.total_updates = 0
    
    def joint_state_to_key(self, joint_state):
        state_tuples = tuple(tuple(s.astype(int)) for s in joint_state)
        return state_tuples
    
    def joint_action_to_index(self, joint_action):
        index = 0
        for i, action in enumerate(joint_action):
            index += action * (self.action_space_size ** (self.num_agents - 1 - i))
        return index
    
    def index_to_joint_action(self, index):
        joint_action = []
        remaining = index
        for i in range(self.num_agents):
            power = self.action_space_size ** (self.num_agents - 1 - i)
            action = remaining // power
            remaining = remaining % power
            joint_action.append(action)
        return tuple(joint_action)
    
    def select_joint_action(self, joint_state, training=True):
        joint_state_key = self.joint_state_to_key(joint_state)
        if training and self.rng.random() < self.epsilon:
            joint_action = tuple(self.rng.integers(0, self.action_space_size) 
                                for _ in range(self.num_agents))
        else:
            q_values = self.q_table[joint_state_key]
            max_q = np.max(q_values)
            best_indices = np.where(q_values == max_q)[0]
            best_index = int(self.rng.choice(best_indices))
            joint_action = self.index_to_joint_action(best_index)
        return joint_action
    
    def update(self, joint_state, joint_action, team_reward, next_joint_state, done):
        joint_state_key = self.joint_state_to_key(joint_state)
        next_joint_state_key = self.joint_state_to_key(next_joint_state)
        joint_action_idx = self.joint_action_to_index(joint_action)
        current_q = self.q_table[joint_state_key][joint_action_idx]
        if done:
            target_q = team_reward
        else:
            next_max_q = np.max(self.q_table[next_joint_state_key])
            target_q = team_reward + self.discount_factor * next_max_q
        new_q = current_q + self.learning_rate * (target_q - current_q)
        self.q_table[joint_state_key][joint_action_idx] = new_q
        self.total_updates += 1
    
    def decay_epsilon(self, episode):
        """指数衰减epsilon"""
        self.epsilon = max(self.epsilon_end, 
                          self.epsilon_start * math.exp(-self.epsilon_decay * episode))
    
    def get_policy(self, joint_state):
        joint_state_key = self.joint_state_to_key(joint_state)
        q_values = self.q_table[joint_state_key]
        best_index = int(np.argmax(q_values))
        return self.index_to_joint_action(best_index)


# ============================================
# 第五部分：训练和评估函数
# ============================================


In [6]:
def train_independent_qlearning(env, num_episodes=2000, max_steps_per_episode=200,
                                learning_rate=0.1, discount_factor=0.95,
                                epsilon_start=1.0, epsilon_end=0.1, epsilon_decay=0.995, seed=42):
    """训练Independent Q-learning"""
    num_agents = env.num_agents
    state_space_size = (env.rows, env.cols, 2)
    action_space_size = 6
    
    agent = IndependentQLearning(
        num_agents=num_agents,
        state_space_size=state_space_size,
        action_space_size=action_space_size,
        learning_rate=learning_rate,
        discount_factor=discount_factor,
        epsilon_start=epsilon_start,
        epsilon_end=epsilon_end,
        epsilon_decay=epsilon_decay
    )
    
    episode_rewards = []
    episode_epsilons = []  # 记录每个episode的epsilon值
    episode_lengths = []
    episode_success = []
    recent_rewards = deque(maxlen=100)
    
    print("Training Independent Q-Learning...")
    print(f"Total episodes: {num_episodes}")
    
    for episode in range(num_episodes):
        observations, info = env.reset(seed=seed)
        episode_reward = [0.0 for _ in range(num_agents)]
        episode_length = 0
        done = False
        
        while not done and episode_length < max_steps_per_episode:
            actions = []
            for i in range(num_agents):
                action = agent.select_action(i, observations[i], training=True)
                actions.append(action)
            
            next_observations, rewards, terminated, truncated, info = env.step(tuple(actions))
            done = terminated or truncated
            
            for i in range(num_agents):
                agent.update(
                    agent_idx=i,
                    state=observations[i],
                    action=actions[i],
                    reward=rewards[i],
                    next_state=next_observations[i],
                    done=done
                )
                episode_reward[i] += rewards[i]
            
            observations = next_observations
            episode_length += 1
        
        total_episode_reward = sum(episode_reward)
        episode_rewards.append(total_episode_reward)
        episode_lengths.append(episode_length)
        recent_rewards.append(total_episode_reward)
        success = all(info['agent_delivered'])
        episode_success.append(1 if success else 0)
        agent.decay_epsilon(episode)  # 使用指数衰减，传入episode
        episode_epsilons.append(agent.epsilon)  # 记录epsilon值
        
        if (episode + 1) % 100 == 0:
            avg_reward = np.mean(recent_rewards)
            success_rate = np.mean(episode_success[-100:])
            print(f"Episode {episode + 1}/{num_episodes} | "
                  f"Avg Reward: {avg_reward:.2f} | "
                  f"Success Rate: {success_rate:.2%} | "
                  f"Epsilon: {agent.epsilon:.3f}")
    
    training_history = {
        'episode_rewards': episode_rewards,
        'episode_lengths': episode_lengths,
        'episode_success': episode_success,
        'episode_epsilons': episode_epsilons,  # 添加epsilon历史
        'num_episodes': num_episodes
    }
    
    return training_history, agent


In [7]:
def train_cooperative_qlearning(env, num_episodes=2000, max_steps_per_episode=200,
                                learning_rate=0.1, discount_factor=0.95,
                                epsilon_start=1.0, epsilon_end=0.1, epsilon_decay=0.995, seed=42):
    """训练Cooperative Q-learning"""
    num_agents = env.num_agents
    state_space_size = (env.rows, env.cols, 2)
    action_space_size = 6
    
    agent = CooperativeQLearning(
        num_agents=num_agents,
        state_space_size=state_space_size,
        action_space_size=action_space_size,
        learning_rate=learning_rate,
        discount_factor=discount_factor,
        epsilon_start=epsilon_start,
        epsilon_end=epsilon_end,
        epsilon_decay=epsilon_decay,
        seed=seed
    )
    
    episode_rewards = []
    episode_epsilons = []  # 记录每个episode的epsilon值
    episode_lengths = []
    episode_success = []
    recent_rewards = deque(maxlen=100)
    
    print("Training Cooperative Q-Learning...")
    print(f"Total episodes: {num_episodes}")
    
    for episode in range(num_episodes):
        observations, info = env.reset(seed=seed)
        episode_reward = 0.0
        episode_length = 0
        done = False
        
        while not done and episode_length < max_steps_per_episode:
            joint_action = agent.select_joint_action(observations, training=True)
            next_observations, rewards, terminated, truncated, info = env.step(joint_action)
            done = terminated or truncated
            team_reward = sum(rewards)
            episode_reward += team_reward
            
            agent.update(
                joint_state=observations,
                joint_action=joint_action,
                team_reward=team_reward,
                next_joint_state=next_observations,
                done=done
            )
            
            observations = next_observations
            episode_length += 1
        
        episode_rewards.append(episode_reward)
        episode_lengths.append(episode_length)
        recent_rewards.append(episode_reward)
        success = all(info['agent_delivered'])
        episode_success.append(1 if success else 0)
        agent.decay_epsilon(episode)  # 使用指数衰减，传入episode
        episode_epsilons.append(agent.epsilon)  # 记录epsilon值
        
        if (episode + 1) % 100 == 0:
            avg_reward = np.mean(recent_rewards)
            success_rate = np.mean(episode_success[-100:])
            print(f"Episode {episode + 1}/{num_episodes} | "
                  f"Avg Reward: {avg_reward:.2f} | "
                  f"Success Rate: {success_rate:.2%} | "
                  f"Epsilon: {agent.epsilon:.3f}")
    
    training_history = {
        'episode_rewards': episode_rewards,
        'episode_lengths': episode_lengths,
        'episode_success': episode_success,
        'episode_epsilons': episode_epsilons,  # 添加epsilon历史
        'num_episodes': num_episodes
    }
    
    return training_history, agent


In [8]:
def evaluate_agent(env, agent, method='independent', num_episodes=100, max_steps=200, seed=None):
    """评估训练好的智能体"""
    episode_rewards = []
    episode_lengths = []
    episode_success = []
    
    print(f"\nEvaluating {method} agent...")
    
    # 如果提供了seed，为每个episode使用不同的seed（seed + episode）
    # 这样可以测试策略在不同初始状态下的表现
    for episode in range(num_episodes):
        if seed is not None:
            episode_seed = seed + episode  # 每个episode使用不同的seed
        else:
            episode_seed = None
        observations, info = env.reset(seed=episode_seed)
        episode_reward = 0.0
        episode_length = 0
        done = False
        
        while not done and episode_length < max_steps:
            if method == 'independent':
                actions = []
                for i in range(env.num_agents):
                    action = agent.select_action(i, observations[i], training=False)
                    actions.append(action)
                actions = tuple(actions)
            else:
                joint_action = agent.select_joint_action(observations, training=False)
                actions = joint_action
            
            next_observations, rewards, terminated, truncated, info = env.step(actions)
            done = terminated or truncated
            episode_reward += sum(rewards)
            observations = next_observations
            episode_length += 1
        
        episode_rewards.append(episode_reward)
        episode_lengths.append(episode_length)
        success = all(info['agent_delivered'])
        episode_success.append(1 if success else 0)
    
    evaluation_results = {
        'episode_rewards': episode_rewards,
        'episode_lengths': episode_lengths,
        'episode_success': episode_success,
        'mean_reward': np.mean(episode_rewards),
        'std_reward': np.std(episode_rewards),
        'mean_length': np.mean(episode_lengths),
        'std_length': np.std(episode_lengths),
        'success_rate': np.mean(episode_success)
    }
    
    print(f"Mean Reward: {evaluation_results['mean_reward']:.2f} ± {evaluation_results['std_reward']:.2f}")
    print(f"Mean Length: {evaluation_results['mean_length']:.2f} ± {evaluation_results['std_length']:.2f}")
    print(f"Success Rate: {evaluation_results['success_rate']:.2%}")
    
    return evaluation_results


In [9]:
def plot_training_curves(history_iql, history_cql, save_path='results'):
    """绘制训练曲线"""
    os.makedirs(save_path, exist_ok=True)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # 1. 奖励曲线
    ax = axes[0, 0]
    window = 50
    if len(history_iql['episode_rewards']) >= window:
        smoothed_iql = np.convolve(history_iql['episode_rewards'], 
                                   np.ones(window)/window, mode='valid')
        smoothed_cql = np.convolve(history_cql['episode_rewards'],
                                   np.ones(window)/window, mode='valid')
        ax.plot(smoothed_iql, label='Independent Q-Learning', alpha=0.7)
        ax.plot(smoothed_cql, label='Cooperative Q-Learning', alpha=0.7)
    else:
        ax.plot(history_iql['episode_rewards'], 
               label='Independent Q-Learning', alpha=0.7)
        ax.plot(history_cql['episode_rewards'],
               label='Cooperative Q-Learning', alpha=0.7)
    
    ax.set_xlabel('Episode')
    ax.set_ylabel('Total Reward')
    ax.set_title('Training: Episode Rewards')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. 成功率曲线
    ax = axes[0, 1]
    if len(history_iql['episode_success']) >= window:
        success_iql = np.convolve(history_iql['episode_success'],
                                 np.ones(window)/window, mode='valid')
        success_cql = np.convolve(history_cql['episode_success'],
                                 np.ones(window)/window, mode='valid')
        ax.plot(success_iql, label='Independent Q-Learning', alpha=0.7)
        ax.plot(success_cql, label='Cooperative Q-Learning', alpha=0.7)
    else:
        ax.plot(history_iql['episode_success'],
               label='Independent Q-Learning', alpha=0.7)
        ax.plot(history_cql['episode_success'],
               label='Cooperative Q-Learning', alpha=0.7)
    
    ax.set_xlabel('Episode')
    ax.set_ylabel('Success Rate')
    ax.set_title('Training: Success Rate')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. Epsilon衰减曲线（Exploration-Exploitation）
    ax = axes[0, 2]
    if 'episode_epsilons' in history_iql and len(history_iql['episode_epsilons']) > 0:
        ax.plot(history_iql['episode_epsilons'], 
               label='Independent Q-Learning', alpha=0.7, color='blue')
    if 'episode_epsilons' in history_cql and len(history_cql['episode_epsilons']) > 0:
        ax.plot(history_cql['episode_epsilons'], 
               label='Cooperative Q-Learning', alpha=0.7, color='orange')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Epsilon (Exploration Rate)')
    ax.set_title('Exploration Rate Decay')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0.1, color='r', linestyle='--', alpha=0.5, label='Low Exploration Threshold')
    
    # 4. 步数曲线
    ax = axes[1, 0]
    if len(history_iql['episode_lengths']) >= window:
        length_iql = np.convolve(history_iql['episode_lengths'],
                                np.ones(window)/window, mode='valid')
        length_cql = np.convolve(history_cql['episode_lengths'],
                                np.ones(window)/window, mode='valid')
        ax.plot(length_iql, label='Independent Q-Learning', alpha=0.7)
        ax.plot(length_cql, label='Cooperative Q-Learning', alpha=0.7)
    else:
        ax.plot(history_iql['episode_lengths'],
               label='Independent Q-Learning', alpha=0.7)
        ax.plot(history_cql['episode_lengths'],
               label='Cooperative Q-Learning', alpha=0.7)
    
    ax.set_xlabel('Episode')
    ax.set_ylabel('Episode Length')
    ax.set_title('Training: Episode Lengths')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 5. 奖励分布（最后100轮）
    ax = axes[1, 1]
    final_rewards_iql = history_iql['episode_rewards'][-100:]
    final_rewards_cql = history_cql['episode_rewards'][-100:]
    ax.hist(final_rewards_iql, bins=20, alpha=0.5, label='Independent Q-Learning')
    ax.hist(final_rewards_cql, bins=20, alpha=0.5, label='Cooperative Q-Learning')
    ax.set_xlabel('Total Reward')
    ax.set_ylabel('Frequency')
    ax.set_title('Final 100 Episodes: Reward Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 6. Exploration vs Exploitation分析
    ax = axes[1, 2]
    if 'episode_epsilons' in history_iql and len(history_iql['episode_epsilons']) > 0:
        # 计算exploration和exploitation的比例
        epsilons_iql = np.array(history_iql['episode_epsilons'])
        exploration_ratio_iql = epsilons_iql  # exploration比例 = epsilon
        exploitation_ratio_iql = 1 - epsilons_iql  # exploitation比例 = 1 - epsilon
        
        # 使用滑动窗口平滑
        window_size = 100
        if len(exploration_ratio_iql) >= window_size:
            exploration_smooth_iql = np.convolve(exploration_ratio_iql, 
                                                 np.ones(window_size)/window_size, mode='valid')
            exploitation_smooth_iql = np.convolve(exploitation_ratio_iql, 
                                                  np.ones(window_size)/window_size, mode='valid')
            x_axis = np.arange(len(exploration_smooth_iql))
        else:
            exploration_smooth_iql = exploration_ratio_iql
            exploitation_smooth_iql = exploitation_ratio_iql
            x_axis = np.arange(len(exploration_smooth_iql))
        
        ax.plot(x_axis, exploration_smooth_iql, label='Exploration (IQL)', alpha=0.7, color='blue')
        ax.plot(x_axis, exploitation_smooth_iql, label='Exploitation (IQL)', alpha=0.7, color='lightblue')
    
    if 'episode_epsilons' in history_cql and len(history_cql['episode_epsilons']) > 0:
        epsilons_cql = np.array(history_cql['episode_epsilons'])
        exploration_ratio_cql = epsilons_cql
        exploitation_ratio_cql = 1 - epsilons_cql
        
        window_size = 100
        if len(exploration_ratio_cql) >= window_size:
            exploration_smooth_cql = np.convolve(exploration_ratio_cql, 
                                                 np.ones(window_size)/window_size, mode='valid')
            exploitation_smooth_cql = np.convolve(exploitation_ratio_cql, 
                                                  np.ones(window_size)/window_size, mode='valid')
            x_axis = np.arange(len(exploration_smooth_cql))
        else:
            exploration_smooth_cql = exploration_ratio_cql
            exploitation_smooth_cql = exploitation_ratio_cql
            x_axis = np.arange(len(exploration_smooth_cql))
        
        ax.plot(x_axis, exploration_smooth_cql, label='Exploration (CQL)', alpha=0.7, color='orange', linestyle='--')
        ax.plot(x_axis, exploitation_smooth_cql, label='Exploitation (CQL)', alpha=0.7, color='peachpuff', linestyle='--')
    
    ax.set_xlabel('Episode')
    ax.set_ylabel('Ratio')
    ax.set_title('Exploration vs Exploitation Balance')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, 'training_curves.png'), dpi=300)
    print(f"\nTraining curves saved to {os.path.join(save_path, 'training_curves.png')}")
    plt.close()
    """绘制训练曲线"""
    os.makedirs(save_path, exist_ok=True)
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # 1. 奖励曲线
    ax = axes[0, 0]
    window = 50
    if len(history_iql['episode_rewards']) >= window:
        smoothed_iql = np.convolve(history_iql['episode_rewards'], 
                                   np.ones(window)/window, mode='valid')
        smoothed_cql = np.convolve(history_cql['episode_rewards'],
                                   np.ones(window)/window, mode='valid')
        ax.plot(smoothed_iql, label='Independent Q-Learning', alpha=0.7)
        ax.plot(smoothed_cql, label='Cooperative Q-Learning', alpha=0.7)
    else:
        ax.plot(history_iql['episode_rewards'], 
               label='Independent Q-Learning', alpha=0.7)
        ax.plot(history_cql['episode_rewards'],
               label='Cooperative Q-Learning', alpha=0.7)
    
    ax.set_xlabel('Episode')
    ax.set_ylabel('Total Reward')
    ax.set_title('Training: Episode Rewards')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. 成功率曲线
    ax = axes[0, 1]
    if len(history_iql['episode_success']) >= window:
        success_iql = np.convolve(history_iql['episode_success'],
                                 np.ones(window)/window, mode='valid')
        success_cql = np.convolve(history_cql['episode_success'],
                                 np.ones(window)/window, mode='valid')
        ax.plot(success_iql, label='Independent Q-Learning', alpha=0.7)
        ax.plot(success_cql, label='Cooperative Q-Learning', alpha=0.7)
    else:
        ax.plot(history_iql['episode_success'],
               label='Independent Q-Learning', alpha=0.7)
        ax.plot(history_cql['episode_success'],
               label='Cooperative Q-Learning', alpha=0.7)
    
    ax.set_xlabel('Episode')
    ax.set_ylabel('Success Rate')
    ax.set_title('Training: Success Rate')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. 步数曲线
    ax = axes[1, 0]
    if len(history_iql['episode_lengths']) >= window:
        length_iql = np.convolve(history_iql['episode_lengths'],
                                np.ones(window)/window, mode='valid')
        length_cql = np.convolve(history_cql['episode_lengths'],
                                np.ones(window)/window, mode='valid')
        ax.plot(length_iql, label='Independent Q-Learning', alpha=0.7)
        ax.plot(length_cql, label='Cooperative Q-Learning', alpha=0.7)
    else:
        ax.plot(history_iql['episode_lengths'],
               label='Independent Q-Learning', alpha=0.7)
        ax.plot(history_cql['episode_lengths'],
               label='Cooperative Q-Learning', alpha=0.7)
    
    ax.set_xlabel('Episode')
    ax.set_ylabel('Episode Length')
    ax.set_title('Training: Episode Lengths')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 4. 奖励分布（最后100轮）
    ax = axes[1, 1]
    final_rewards_iql = history_iql['episode_rewards'][-100:]
    final_rewards_cql = history_cql['episode_rewards'][-100:]
    ax.hist(final_rewards_iql, bins=20, alpha=0.5, label='Independent Q-Learning')
    ax.hist(final_rewards_cql, bins=20, alpha=0.5, label='Cooperative Q-Learning')
    ax.set_xlabel('Total Reward')
    ax.set_ylabel('Frequency')
    ax.set_title('Final 100 Episodes: Reward Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, 'training_curves.png'), dpi=300)
    print(f"\nTraining curves saved to {os.path.join(save_path, 'training_curves.png')}")
    plt.close()


In [10]:
def analyze_exploration_exploitation(history_iql, history_cql):
    """分析Exploration-Exploitation平衡"""
    print("\n" + "=" * 60)
    print("Exploration-Exploitation Analysis")
    print("=" * 60)
    
    if 'episode_epsilons' not in history_iql or len(history_iql['episode_epsilons']) == 0:
        print("Warning: No epsilon data found in Independent Q-Learning history")
        return
    
    epsilons_iql = np.array(history_iql['episode_epsilons'])
    epsilons_cql = np.array(history_cql['episode_epsilons']) if 'episode_epsilons' in history_cql else None
    
    # 计算不同阶段的exploration比例
    total_episodes = len(epsilons_iql)
    early_stage = int(total_episodes * 0.1)  # 前10%
    mid_stage = int(total_episodes * 0.5)    # 前50%
    late_stage = int(total_episodes * 0.9)   # 前90%
    
    print("\nIndependent Q-Learning:")
    print(f"  Initial Epsilon: {epsilons_iql[0]:.4f}")
    print(f"  Final Epsilon: {epsilons_iql[-1]:.4f}")
    print(f"  Early Stage (0-{early_stage}): Avg Epsilon = {np.mean(epsilons_iql[:early_stage]):.4f}")
    print(f"  Mid Stage (0-{mid_stage}): Avg Epsilon = {np.mean(epsilons_iql[:mid_stage]):.4f}")
    print(f"  Late Stage (0-{late_stage}): Avg Epsilon = {np.mean(epsilons_iql[:late_stage]):.4f}")
    print(f"  Final Stage ({late_stage}-{total_episodes}): Avg Epsilon = {np.mean(epsilons_iql[late_stage:]):.4f}")
    
    # 计算exploration和exploitation的时间点
    exploration_threshold = 0.5  # epsilon > 0.5 认为是exploration
    exploitation_threshold = 0.1  # epsilon < 0.1 认为是exploitation
    
    exploration_episodes_iql = np.sum(epsilons_iql > exploration_threshold)
    exploitation_episodes_iql = np.sum(epsilons_iql < exploitation_threshold)
    transition_episodes_iql = total_episodes - exploration_episodes_iql - exploitation_episodes_iql
    
    print(f"\n  Exploration Phase (ε > {exploration_threshold}): {exploration_episodes_iql} episodes ({100*exploration_episodes_iql/total_episodes:.1f}%)")
    print(f"  Transition Phase ({exploitation_threshold} ≤ ε ≤ {exploration_threshold}): {transition_episodes_iql} episodes ({100*transition_episodes_iql/total_episodes:.1f}%)")
    print(f"  Exploitation Phase (ε < {exploitation_threshold}): {exploitation_episodes_iql} episodes ({100*exploitation_episodes_iql/total_episodes:.1f}%)")
    
    if epsilons_cql is not None and len(epsilons_cql) > 0:
        print("\nCooperative Q-Learning:")
        print(f"  Initial Epsilon: {epsilons_cql[0]:.4f}")
        print(f"  Final Epsilon: {epsilons_cql[-1]:.4f}")
        print(f"  Early Stage (0-{early_stage}): Avg Epsilon = {np.mean(epsilons_cql[:early_stage]):.4f}")
        print(f"  Mid Stage (0-{mid_stage}): Avg Epsilon = {np.mean(epsilons_cql[:mid_stage]):.4f}")
        print(f"  Late Stage (0-{late_stage}): Avg Epsilon = {np.mean(epsilons_cql[:late_stage]):.4f}")
        print(f"  Final Stage ({late_stage}-{total_episodes}): Avg Epsilon = {np.mean(epsilons_cql[late_stage:]):.4f}")
        
        exploration_episodes_cql = np.sum(epsilons_cql > exploration_threshold)
        exploitation_episodes_cql = np.sum(epsilons_cql < exploitation_threshold)
        transition_episodes_cql = total_episodes - exploration_episodes_cql - exploitation_episodes_cql
        
        print(f"\n  Exploration Phase (ε > {exploration_threshold}): {exploration_episodes_cql} episodes ({100*exploration_episodes_cql/total_episodes:.1f}%)")
        print(f"  Transition Phase ({exploitation_threshold} ≤ ε ≤ {exploration_threshold}): {transition_episodes_cql} episodes ({100*transition_episodes_cql/total_episodes:.1f}%)")
        print(f"  Exploitation Phase (ε < {exploitation_threshold}): {exploitation_episodes_cql} episodes ({100*exploitation_episodes_cql/total_episodes:.1f}%)")
    
    print("\n" + "=" * 60)


# ============================================
# 第六部分：运行完整实验
# ============================================


In [11]:
# 创建多智能体环境
config = WarehouseConfig()
num_agents = 2  # 改为2个智能体，减少联合动作空间（从6^3=216降到6^2=36）
env = MultiAgentWarehouseEnv(
    base_env_class=WarehouseRobotDetEnv,
    config=config,
    num_agents=num_agents,
    seed=42
)

print(f"Environment created with {num_agents} agents")
print(f"Grid size: {env.rows}x{env.cols}")
print(f"Joint action space size: {6**num_agents} (for Cooperative Q-Learning)")


Environment created with 2 agents
Grid size: 6x6
Joint action space size: 36 (for Cooperative Q-Learning)


In [12]:
# 训练参数（进一步优化 - 提高稳定性）
num_episodes = 5000  # 增加到5000轮，确保充分学习
max_steps_per_episode = 200
learning_rate = 0.15  # 降低学习率，提高稳定性
discount_factor = 0.95
epsilon_start = 1.0
epsilon_end = 0.05
epsilon_decay = 0.00092  # 指数衰减率：ε = ε_start * e^(-decay_rate * episode)
# 让epsilon在训练过程中逐渐衰减：
# - Episode 1000: ε ≈ 0.40
# - Episode 2500: ε ≈ 0.10
# - Episode 5000: ε ≈ 0.01（会被限制为epsilon_end=0.05）

print("=" * 60)
print("Checkpoint 1: Multi-Agent Reinforcement Learning")
print("Comparing Independent Q-Learning vs Cooperative Q-Learning")
print(f"Number of agents: {num_agents}")
print(f"Training episodes: {num_episodes}")
print(f"Epsilon decay: exponential with rate 0.00092")
print("=" * 60)


Checkpoint 1: Multi-Agent Reinforcement Learning
Comparing Independent Q-Learning vs Cooperative Q-Learning
Number of agents: 2
Training episodes: 5000
Epsilon decay: exponential with rate 0.00092


In [13]:
# 训练Independent Q-Learning
print("\n" + "=" * 60)
print("Training Independent Q-Learning")
print("=" * 60)

history_iql, agent_iql = train_independent_qlearning(
    env=env,
    num_episodes=num_episodes,
    max_steps_per_episode=max_steps_per_episode,
    learning_rate=learning_rate,
    discount_factor=discount_factor,
    epsilon_start=epsilon_start,
    epsilon_end=epsilon_end,
    epsilon_decay=epsilon_decay
)



Training Independent Q-Learning
Training Independent Q-Learning...
Total episodes: 5000
Episode 100/5000 | Avg Reward: -867.73 | Success Rate: 5.00% | Epsilon: 0.913
Episode 200/5000 | Avg Reward: -375.46 | Success Rate: 61.00% | Epsilon: 0.833
Episode 300/5000 | Avg Reward: -18.17 | Success Rate: 96.00% | Epsilon: 0.760
Episode 400/5000 | Avg Reward: 147.77 | Success Rate: 100.00% | Epsilon: 0.693
Episode 500/5000 | Avg Reward: 235.62 | Success Rate: 100.00% | Epsilon: 0.632
Episode 600/5000 | Avg Reward: 281.03 | Success Rate: 100.00% | Epsilon: 0.576
Episode 700/5000 | Avg Reward: 313.74 | Success Rate: 100.00% | Epsilon: 0.526
Episode 800/5000 | Avg Reward: 339.35 | Success Rate: 100.00% | Epsilon: 0.479
Episode 900/5000 | Avg Reward: 359.56 | Success Rate: 100.00% | Epsilon: 0.437
Episode 1000/5000 | Avg Reward: 370.13 | Success Rate: 100.00% | Epsilon: 0.399
Episode 1100/5000 | Avg Reward: 371.00 | Success Rate: 100.00% | Epsilon: 0.364
Episode 1200/5000 | Avg Reward: 384.90 | S

In [14]:
# 训练Cooperative Q-Learning
print("\n" + "=" * 60)
print("Training Cooperative Q-Learning")
print("=" * 60)

history_cql, agent_cql = train_cooperative_qlearning(
    env=env,
    num_episodes=num_episodes,
    max_steps_per_episode=max_steps_per_episode,
    learning_rate=learning_rate,
    discount_factor=discount_factor,
    epsilon_start=epsilon_start,
    epsilon_end=epsilon_end,
    epsilon_decay=epsilon_decay
)



Training Cooperative Q-Learning
Training Cooperative Q-Learning...
Total episodes: 5000
Episode 100/5000 | Avg Reward: -985.45 | Success Rate: 0.00% | Epsilon: 0.913
Episode 200/5000 | Avg Reward: -943.99 | Success Rate: 1.00% | Epsilon: 0.833
Episode 300/5000 | Avg Reward: -910.54 | Success Rate: 1.00% | Epsilon: 0.760
Episode 400/5000 | Avg Reward: -750.63 | Success Rate: 4.00% | Epsilon: 0.693
Episode 500/5000 | Avg Reward: -608.96 | Success Rate: 8.00% | Epsilon: 0.632
Episode 600/5000 | Avg Reward: -465.38 | Success Rate: 20.00% | Epsilon: 0.576
Episode 700/5000 | Avg Reward: -188.79 | Success Rate: 54.00% | Epsilon: 0.526
Episode 800/5000 | Avg Reward: 188.30 | Success Rate: 92.00% | Epsilon: 0.479
Episode 900/5000 | Avg Reward: 282.41 | Success Rate: 99.00% | Epsilon: 0.437
Episode 1000/5000 | Avg Reward: 321.98 | Success Rate: 100.00% | Epsilon: 0.399
Episode 1100/5000 | Avg Reward: 325.68 | Success Rate: 98.00% | Epsilon: 0.364
Episode 1200/5000 | Avg Reward: 354.37 | Success

In [15]:
# 评估两种方法
print("\n" + "=" * 60)
print("Evaluation")
print("=" * 60)

eval_iql = evaluate_agent(env, agent_iql, method='independent', num_episodes=100, seed=42)
eval_cql = evaluate_agent(env, agent_cql, method='cooperative', num_episodes=100, seed=42)



Evaluation

Evaluating independent agent...
Mean Reward: 74.75 ± 348.50
Mean Length: 115.78 ± 91.25
Success Rate: 46.00%

Evaluating cooperative agent...
Mean Reward: -255.23 ± 612.62
Mean Length: 125.49 ± 79.67
Success Rate: 48.00%


In [16]:
# Exploration-Exploitation分析
analyze_exploration_exploitation(history_iql, history_cql)



Exploration-Exploitation Analysis

Independent Q-Learning:
  Initial Epsilon: 1.0000
  Final Epsilon: 0.0500
  Early Stage (0-500): Avg Epsilon = 0.8019
  Mid Stage (0-2500): Avg Epsilon = 0.3914
  Late Stage (0-4500): Avg Epsilon = 0.2434
  Final Stage (4500-5000): Avg Epsilon = 0.0500

  Exploration Phase (ε > 0.5): 754 episodes (15.1%)
  Transition Phase (0.1 ≤ ε ≤ 0.5): 1749 episodes (35.0%)
  Exploitation Phase (ε < 0.1): 2497 episodes (49.9%)

Cooperative Q-Learning:
  Initial Epsilon: 1.0000
  Final Epsilon: 0.0500
  Early Stage (0-500): Avg Epsilon = 0.8019
  Mid Stage (0-2500): Avg Epsilon = 0.3914
  Late Stage (0-4500): Avg Epsilon = 0.2434
  Final Stage (4500-5000): Avg Epsilon = 0.0500

  Exploration Phase (ε > 0.5): 754 episodes (15.1%)
  Transition Phase (0.1 ≤ ε ≤ 0.5): 1749 episodes (35.0%)
  Exploitation Phase (ε < 0.1): 2497 episodes (49.9%)



In [17]:
# 绘制结果
print("\n" + "=" * 60)
print("Generating plots...")
print("=" * 60)

os.makedirs('results', exist_ok=True)
plot_training_curves(history_iql, history_cql, save_path='results')



Generating plots...

Training curves saved to results\training_curves.png

Training curves saved to results\training_curves.png


In [18]:
# 保存结果
import pickle
results = {
    'independent': {
        'training_history': history_iql,
        'evaluation': eval_iql
    },
    'cooperative': {
        'training_history': history_cql,
        'evaluation': eval_cql
    }
}

with open('results/experiment_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print("Results saved to results/experiment_results.pkl")


Results saved to results/experiment_results.pkl


In [19]:
# 打印总结
print("\n" + "=" * 60)
print("Summary")
print("=" * 60)
print(f"\nIndependent Q-Learning:")
print(f"  Final Mean Reward: {eval_iql['mean_reward']:.2f} ± {eval_iql['std_reward']:.2f}")
print(f"  Success Rate: {eval_iql['success_rate']:.2%}")
print(f"  Mean Episode Length: {eval_iql['mean_length']:.2f} ± {eval_iql['std_length']:.2f}")

print(f"\nCooperative Q-Learning:")
print(f"  Final Mean Reward: {eval_cql['mean_reward']:.2f} ± {eval_cql['std_reward']:.2f}")
print(f"  Success Rate: {eval_cql['success_rate']:.2%}")
print(f"  Mean Episode Length: {eval_cql['mean_length']:.2f} ± {eval_cql['std_length']:.2f}")

print("\n" + "=" * 60)
print("Experiment completed!")
print("=" * 60)

env.close()



Summary

Independent Q-Learning:
  Final Mean Reward: 74.75 ± 348.50
  Success Rate: 46.00%
  Mean Episode Length: 115.78 ± 91.25

Cooperative Q-Learning:
  Final Mean Reward: -255.23 ± 612.62
  Success Rate: 48.00%
  Mean Episode Length: 125.49 ± 79.67

Experiment completed!
